# Figure 3 calculation-only notebook

This notebook **does not create figures**.

It computes and saves the CSV files needed for Figure 3:

- training-only differential clinical feature analysis
- held-out train/test effect-size validation
- CoxPH support analysis using the `cox/` folder
- ML top-30 overlap/support matrix

Expected folder layout:

```text
differential_analysis/
├── code/
│   └── 01_compute_figure3_results_only_v6_with_cox_folder.ipynb
├── data/
│   ├── X_train.csv
│   ├── X_test.csv
│   ├── y_train.csv
│   ├── y_test.csv
│   ├── top30_feature_rankings_long.csv
│   ├── top30_overlap_summary.csv
│   └── top30_shared_features_by_pair.csv
├── cox/
│   ├── cox_input_train_ab_valid_rows.csv
│   ├── cox_input_test_c_valid_rows.csv
│   └── cox_input_valid_rows.csv
└── result/
```

All outputs are saved to:

```text
result/figure3_calculation/
```


In [1]:
# ============================================================
# 1. Imports and global settings
# ============================================================

from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd

from scipy import stats

try:
    from statsmodels.stats.multitest import multipletests
    HAS_STATSMODELS_MULTITEST = True
except Exception:
    HAS_STATSMODELS_MULTITEST = False

# CoxPH backend: prefer lifelines, fallback to statsmodels PHReg.
try:
    from lifelines import CoxPHFitter
    HAS_LIFELINES = True
except Exception:
    HAS_LIFELINES = False

try:
    from statsmodels.duration.hazard_regression import PHReg
    HAS_PHREG = True
except Exception:
    HAS_PHREG = False

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
DIFF_FDR_THRESHOLD = 0.05
DIFF_ABS_SMD_THRESHOLD = 0.20
COX_FDR_THRESHOLD = 0.05

print("Package availability:")
print(f"  lifelines CoxPHFitter: {HAS_LIFELINES}")
print(f"  statsmodels PHReg:     {HAS_PHREG}")
print(f"  statsmodels multitest: {HAS_STATSMODELS_MULTITEST}")


Package availability:
  lifelines CoxPHFitter: False
  statsmodels PHReg:     False
  statsmodels multitest: False


In [2]:
# ============================================================
# 2. Locate project folders
# ============================================================

def find_project_dir(start: Path) -> Path:
    """
    If this notebook is run from differential_analysis/code, use the parent folder.
    Otherwise, walk upward until a folder containing data/ is found.
    """
    start = Path(start).resolve()

    if start.name.lower() == "code" and (start.parent / "data").exists():
        return start.parent

    if (start / "data").exists():
        return start

    for parent in [start] + list(start.parents):
        if (parent / "data").exists():
            return parent

    return start.parent if start.name.lower() == "code" else start

PROJECT_DIR = find_project_dir(Path.cwd())
CODE_DIR = PROJECT_DIR / "code"
DATA_DIR = PROJECT_DIR / "data"
COX_DIR = PROJECT_DIR / "cox"
RESULT_DIR = PROJECT_DIR / "result"
FIG3_RESULT_DIR = RESULT_DIR / "figure3_calculation"

FIG3_RESULT_DIR.mkdir(parents=True, exist_ok=True)

print("Folder paths:")
print(f"  PROJECT_DIR:       {PROJECT_DIR}")
print(f"  DATA_DIR:          {DATA_DIR}")
print(f"  COX_DIR:           {COX_DIR}")
print(f"  FIG3_RESULT_DIR:   {FIG3_RESULT_DIR}")


Folder paths:
  PROJECT_DIR:       C:\Users\junse\Documents\research\IUBDC 2026\differential_analysis
  DATA_DIR:          C:\Users\junse\Documents\research\IUBDC 2026\differential_analysis\data
  COX_DIR:           C:\Users\junse\Documents\research\IUBDC 2026\differential_analysis\cox
  FIG3_RESULT_DIR:   C:\Users\junse\Documents\research\IUBDC 2026\differential_analysis\result\figure3_calculation


In [3]:
# ============================================================
# 3. Robust file loading helpers
# ============================================================

def find_first_existing_file(search_dirs, candidates, required=True, label="file"):
    """
    Search for the first existing filename across multiple directories.
    """
    search_dirs = [Path(d) for d in search_dirs if d is not None]

    tried = []
    for directory in search_dirs:
        for name in candidates:
            path = directory / name
            tried.append(path)
            if path.exists():
                return path

    if required:
        msg = "\n".join(str(p) for p in tried)
        raise FileNotFoundError(
            f"Could not find required {label}. Tried:\n{msg}\n\n"
            "Please put the file in the expected folder or add its filename to the candidate list."
        )
    return None


def read_csv_safely(path, label="file"):
    if path is None:
        return pd.DataFrame()
    df = pd.read_csv(path)
    print(f"Loaded {label}: {path.name} | shape={df.shape}")
    return df


def normalize_colnames(df):
    """
    Normalize common column variants without destroying original feature columns.
    """
    df = df.copy()
    rename = {}
    for c in df.columns:
        lower = c.strip().lower().replace("-", "_").replace(" ", "_")
        if lower in ["in_hospital_death", "inhospital_death", "hospital_death", "death", "mortality"]:
            rename[c] = "in_hospital_death"
        elif lower == "recordid":
            rename[c] = "RecordID"
        elif lower == "length_of_stay":
            rename[c] = "Length_of_stay"
        elif lower == "saps_i":
            rename[c] = "SAPS_I"
    return df.rename(columns=rename)


X_TRAIN_CANDIDATES = [
    "X_train.csv", "X_train_clean.csv", "X_train_ml_ready.csv",
    "X_train(13).csv", "X_train(4).csv"
]

X_TEST_CANDIDATES = [
    "X_test.csv", "X_test_clean.csv", "X_test_ml_ready.csv",
    "X_test(13).csv", "X_test(4).csv"
]

Y_TRAIN_CANDIDATES = [
    "y_train.csv", "y_train_clean.csv", "y_train_ml_ready.csv",
    "time_series_y_train_a_plus_b_train_c_test.csv",
    "time_series_y_train_a_plus_b_train_c_test(6).csv",
    "time_series_y_train_a_plus_b_train_c_test(5).csv",
    "train_outcomes.csv", "outcomes_train.csv", "y_train(13).csv", "y_train(4).csv"
]

Y_TEST_CANDIDATES = [
    "y_test.csv", "y_test_clean.csv", "y_test_ml_ready.csv",
    "time_series_y_test_a_plus_b_train_c_test.csv",
    "time_series_y_test_a_plus_b_train_c_test(5).csv",
    "test_outcomes.csv", "outcomes_test.csv", "y_test(13).csv", "y_test(4).csv"
]

TIME_SERIES_TRAIN_CANDIDATES = [
    "time_series_train_a_plus_b_train_c_test.csv",
    "time_series_train_a_plus_b_train_c_test(6).csv",
    "time_series_train_a_plus_b_train_c_test(5).csv"
]

TIME_SERIES_TEST_CANDIDATES = [
    "time_series_test_a_plus_b_train_c_test.csv",
    "time_series_test_a_plus_b_train_c_test(5).csv"
]

SPLIT_SUMMARY_CANDIDATES = [
    "time_series_split_summary_a_plus_b_train_c_test.csv",
    "time_series_split_summary_a_plus_b_train_c_test(4).csv"
]

TOP30_RANKING_CANDIDATES = ["top30_feature_rankings_long.csv"]
TOP30_OVERLAP_CANDIDATES = ["top30_overlap_summary.csv"]
TOP30_SHARED_CANDIDATES = ["top30_shared_features_by_pair.csv"]

COX_TRAIN_CANDIDATES = [
    "cox_input_train_ab_valid_rows.csv",
    "cox_input_train_valid_rows.csv",
    "cox_train_ab.csv",
    "cox_train.csv"
]

COX_TEST_CANDIDATES = [
    "cox_input_test_c_valid_rows.csv",
    "cox_input_test_valid_rows.csv",
    "cox_test_c.csv",
    "cox_test.csv"
]

COX_ALL_CANDIDATES = [
    "cox_input_valid_rows.csv",
    "cox_outcome_all_rows_with_qc.csv",
    "cox_all.csv"
]

SEARCH_DATA_DIRS = [DATA_DIR, RESULT_DIR, FIG3_RESULT_DIR]
SEARCH_COX_DIRS = [COX_DIR, DATA_DIR, RESULT_DIR, FIG3_RESULT_DIR]

x_train_path = find_first_existing_file(SEARCH_DATA_DIRS, X_TRAIN_CANDIDATES, required=True, label="X_train")
x_test_path = find_first_existing_file(SEARCH_DATA_DIRS, X_TEST_CANDIDATES, required=True, label="X_test")
y_train_path = find_first_existing_file(SEARCH_DATA_DIRS, Y_TRAIN_CANDIDATES, required=True, label="y_train")
y_test_path = find_first_existing_file(SEARCH_DATA_DIRS, Y_TEST_CANDIDATES, required=False, label="y_test")

time_series_train_path = find_first_existing_file(SEARCH_DATA_DIRS, TIME_SERIES_TRAIN_CANDIDATES, required=False, label="time_series_train")
time_series_test_path = find_first_existing_file(SEARCH_DATA_DIRS, TIME_SERIES_TEST_CANDIDATES, required=False, label="time_series_test")
split_summary_path = find_first_existing_file(SEARCH_DATA_DIRS, SPLIT_SUMMARY_CANDIDATES, required=False, label="split_summary")

top30_ranking_path = find_first_existing_file(SEARCH_DATA_DIRS, TOP30_RANKING_CANDIDATES, required=False, label="top30_rankings")
top30_overlap_path = find_first_existing_file(SEARCH_DATA_DIRS, TOP30_OVERLAP_CANDIDATES, required=False, label="top30_overlap")
top30_shared_path = find_first_existing_file(SEARCH_DATA_DIRS, TOP30_SHARED_CANDIDATES, required=False, label="top30_shared")

cox_train_path = find_first_existing_file(SEARCH_COX_DIRS, COX_TRAIN_CANDIDATES, required=False, label="cox_train")
cox_test_path = find_first_existing_file(SEARCH_COX_DIRS, COX_TEST_CANDIDATES, required=False, label="cox_test")
cox_all_path = find_first_existing_file(SEARCH_COX_DIRS, COX_ALL_CANDIDATES, required=False, label="cox_all")

input_check = pd.DataFrame([
    {"input_name": "X_train", "path": str(x_train_path), "found": x_train_path is not None},
    {"input_name": "X_test", "path": str(x_test_path), "found": x_test_path is not None},
    {"input_name": "y_train", "path": str(y_train_path), "found": y_train_path is not None},
    {"input_name": "y_test", "path": str(y_test_path), "found": y_test_path is not None},
    {"input_name": "time_series_train", "path": str(time_series_train_path), "found": time_series_train_path is not None},
    {"input_name": "time_series_test", "path": str(time_series_test_path), "found": time_series_test_path is not None},
    {"input_name": "split_summary", "path": str(split_summary_path), "found": split_summary_path is not None},
    {"input_name": "top30_rankings", "path": str(top30_ranking_path), "found": top30_ranking_path is not None},
    {"input_name": "top30_overlap", "path": str(top30_overlap_path), "found": top30_overlap_path is not None},
    {"input_name": "top30_shared", "path": str(top30_shared_path), "found": top30_shared_path is not None},
    {"input_name": "cox_train", "path": str(cox_train_path), "found": cox_train_path is not None},
    {"input_name": "cox_test", "path": str(cox_test_path), "found": cox_test_path is not None},
    {"input_name": "cox_all", "path": str(cox_all_path), "found": cox_all_path is not None},
])
input_check.to_csv(FIG3_RESULT_DIR / "input_file_check.csv", index=False)

input_check


,input_name,path,found
0,X_train,C:\Users\junse\Documents\research\IUBDC 2026\d...,True
1,X_test,C:\Users\junse\Documents\research\IUBDC 2026\d...,True
2,y_train,C:\Users\junse\Documents\research\IUBDC 2026\d...,True
3,y_test,C:\Users\junse\Documents\research\IUBDC 2026\d...,True
4,time_series_train,C:\Users\junse\Documents\research\IUBDC 2026\d...,True
5,time_series_test,C:\Users\junse\Documents\research\IUBDC 2026\d...,True
6,split_summary,C:\Users\junse\Documents\research\IUBDC 2026\d...,True
7,top30_rankings,C:\Users\junse\Documents\research\IUBDC 2026\d...,True
8,top30_overlap,C:\Users\junse\Documents\research\IUBDC 2026\d...,True
9,top30_shared,C:\Users\junse\Documents\research\IUBDC 2026\d...,True


In [4]:
# ============================================================
# 4. Load input data
# ============================================================

X_train = normalize_colnames(read_csv_safely(x_train_path, "X_train"))
X_test = normalize_colnames(read_csv_safely(x_test_path, "X_test"))
y_train_raw = normalize_colnames(read_csv_safely(y_train_path, "y_train"))
y_test_raw = normalize_colnames(read_csv_safely(y_test_path, "y_test")) if y_test_path is not None else pd.DataFrame()

top30_rankings = normalize_colnames(read_csv_safely(top30_ranking_path, "top30_rankings")) if top30_ranking_path is not None else pd.DataFrame()
top30_overlap = normalize_colnames(read_csv_safely(top30_overlap_path, "top30_overlap")) if top30_overlap_path is not None else pd.DataFrame()
top30_shared = normalize_colnames(read_csv_safely(top30_shared_path, "top30_shared")) if top30_shared_path is not None else pd.DataFrame()

cox_train = normalize_colnames(read_csv_safely(cox_train_path, "cox_train")) if cox_train_path is not None else pd.DataFrame()
cox_test = normalize_colnames(read_csv_safely(cox_test_path, "cox_test")) if cox_test_path is not None else pd.DataFrame()
cox_all = normalize_colnames(read_csv_safely(cox_all_path, "cox_all")) if cox_all_path is not None else pd.DataFrame()

split_summary = read_csv_safely(split_summary_path, "split_summary") if split_summary_path is not None else pd.DataFrame()
if not split_summary.empty:
    split_summary.to_csv(FIG3_RESULT_DIR / "time_series_split_summary_copy.csv", index=False)

print("\nLoaded shapes:")
print(f"  X_train:       {X_train.shape}")
print(f"  X_test:        {X_test.shape}")
print(f"  y_train_raw:   {y_train_raw.shape}")
print(f"  y_test_raw:    {y_test_raw.shape}")
print(f"  top30_shared:  {top30_shared.shape}")
print(f"  cox_train:     {cox_train.shape}")
print(f"  cox_test:      {cox_test.shape}")


Loaded X_train: X_train.csv | shape=(8000, 243)
Loaded X_test: X_test.csv | shape=(4000, 243)
Loaded y_train: y_train.csv | shape=(8000, 1)
Loaded y_test: y_test.csv | shape=(4000, 1)
Loaded top30_rankings: top30_feature_rankings_long.csv | shape=(60, 5)
Loaded top30_overlap: top30_overlap_summary.csv | shape=(1, 5)
Loaded top30_shared: top30_shared_features_by_pair.csv | shape=(12, 6)
Loaded cox_train: cox_input_train_ab_valid_rows.csv | shape=(7886, 14)
Loaded cox_test: cox_input_test_c_valid_rows.csv | shape=(3947, 14)
Loaded cox_all: cox_input_valid_rows.csv | shape=(11833, 14)
Loaded split_summary: time_series_split_summary_a_plus_b_train_c_test.csv | shape=(2, 8)

Loaded shapes:
  X_train:       (8000, 243)
  X_test:        (4000, 243)
  y_train_raw:   (8000, 1)
  y_test_raw:    (4000, 1)
  top30_shared:  (12, 6)
  cox_train:     (7886, 14)
  cox_test:      (3947, 14)


In [5]:
# ============================================================
# 5. Outcome extraction and row alignment
# ============================================================

def find_outcome_column(df):
    candidates = [
        "in_hospital_death", "In-hospital_death", "In_hospital_death",
        "hospital_death", "death", "mortality", "label", "y", "target", "outcome"
    ]
    for c in candidates:
        if c in df.columns:
            return c

    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    binary_cols = []
    for c in numeric_cols:
        vals = pd.Series(df[c]).dropna().unique()
        vals = set(vals.tolist())
        if vals.issubset({0, 1, 0.0, 1.0}):
            binary_cols.append(c)
    if len(binary_cols) == 1:
        return binary_cols[0]
    raise ValueError(f"Could not identify outcome column. Columns found: {df.columns.tolist()}")

OUTCOME_COL_TRAIN = find_outcome_column(y_train_raw)
OUTCOME_COL_TEST = find_outcome_column(y_test_raw) if not y_test_raw.empty else None

print(f"Outcome column train: {OUTCOME_COL_TRAIN}")
print(f"Outcome column test:  {OUTCOME_COL_TEST}")

y_train = y_train_raw[OUTCOME_COL_TRAIN].astype(int).reset_index(drop=True)
y_test = y_test_raw[OUTCOME_COL_TEST].astype(int).reset_index(drop=True) if OUTCOME_COL_TEST is not None else None

if len(X_train) != len(y_train):
    raise ValueError(f"X_train rows ({len(X_train)}) and y_train rows ({len(y_train)}) do not match.")

if y_test is not None and len(X_test) != len(y_test):
    raise ValueError(f"X_test rows ({len(X_test)}) and y_test rows ({len(y_test)}) do not match.")

if "RecordID" not in X_train.columns and "RecordID" in y_train_raw.columns:
    X_train = X_train.copy()
    X_train.insert(0, "RecordID", y_train_raw["RecordID"].values)

if "RecordID" not in X_test.columns and not y_test_raw.empty and "RecordID" in y_test_raw.columns:
    X_test = X_test.copy()
    X_test.insert(0, "RecordID", y_test_raw["RecordID"].values)

print("Outcome prevalence:")
print(f"  train n={len(y_train)}, death rate={y_train.mean():.4f}")
if y_test is not None:
    print(f"  test  n={len(y_test)}, death rate={y_test.mean():.4f}")

outcome_summary = pd.DataFrame([
    {"split": "train", "n": len(y_train), "n_deaths": int(y_train.sum()), "death_rate": float(y_train.mean())},
    {"split": "test", "n": len(y_test) if y_test is not None else np.nan,
     "n_deaths": int(y_test.sum()) if y_test is not None else np.nan,
     "death_rate": float(y_test.mean()) if y_test is not None else np.nan}
])
outcome_summary.to_csv(FIG3_RESULT_DIR / "outcome_summary.csv", index=False)
outcome_summary


Outcome column train: in_hospital_death
Outcome column test:  in_hospital_death
Outcome prevalence:
  train n=8000, death rate=0.1403
  test  n=4000, death rate=0.1462


,split,n,n_deaths,death_rate
0,train,8000,1122,0.14025
1,test,4000,585,0.14625


In [6]:
# ============================================================
# 6. Clinical domain mapping
# ============================================================

def clinical_domain(feature):
    f = str(feature).lower()

    if any(k in f for k in ["gcs"]):
        return "Neurological"
    if any(k in f for k in ["bun", "creatinine", "urine"]):
        return "Renal"
    if any(k in f for k in ["hr", "map", "sysabp", "diasabp", "abp", "nimap", "nisys", "nidias"]):
        return "Cardiovascular"
    if any(k in f for k in ["pao2", "paco2", "fio2", "mechvent", "resp"]):
        return "Respiratory"
    if any(k in f for k in ["lactate", "glucose", "hco3", "ph", "na", "k", "mg"]):
        return "Metabolic"
    if any(k in f for k in ["hct", "wbc", "platelets"]):
        return "Hematologic"
    if any(k in f for k in ["age", "gender", "height", "weight", "icutype"]):
        return "Static / demographic"
    if any(k in f for k in ["count", "missing", "n_rows", "n_valid", "n_unique"]):
        return "Measurement / missingness"
    return "Other"


domain_map = pd.DataFrame({
    "feature": [c for c in X_train.columns if c != "RecordID"]
})
domain_map["clinical_domain"] = domain_map["feature"].map(clinical_domain)
domain_map.to_csv(FIG3_RESULT_DIR / "feature_domain_map.csv", index=False)
domain_map.head()


,feature,clinical_domain
0,BUN_count,Renal
1,BUN_first,Renal
2,BUN_last,Renal
3,BUN_max,Renal
4,BUN_mean,Renal


In [7]:
# ============================================================
# 7. Training-only differential clinical feature analysis
# ============================================================

def benjamini_hochberg(p_values):
    p_values = np.asarray(p_values, dtype=float)
    valid = np.isfinite(p_values)
    q = np.full_like(p_values, fill_value=np.nan, dtype=float)

    if valid.sum() == 0:
        return q

    if HAS_STATSMODELS_MULTITEST:
        q[valid] = multipletests(p_values[valid], method="fdr_bh")[1]
        return q

    p = p_values[valid]
    n = len(p)
    order = np.argsort(p)
    ranked = p[order]
    q_ranked = ranked * n / (np.arange(1, n + 1))
    q_ranked = np.minimum.accumulate(q_ranked[::-1])[::-1]
    q_ranked = np.minimum(q_ranked, 1.0)

    out = np.empty(n)
    out[order] = q_ranked
    q[valid] = out
    return q


def compute_smd(x0, x1):
    """
    Standardized mean difference = mean(death group) - mean(survivor group)
    divided by pooled SD.
    """
    x0 = pd.to_numeric(pd.Series(x0), errors="coerce").dropna()
    x1 = pd.to_numeric(pd.Series(x1), errors="coerce").dropna()

    if len(x0) < 2 or len(x1) < 2:
        return np.nan

    m0, m1 = x0.mean(), x1.mean()
    v0, v1 = x0.var(ddof=1), x1.var(ddof=1)
    pooled = np.sqrt((v0 + v1) / 2.0)

    if not np.isfinite(pooled) or pooled == 0:
        return np.nan

    return (m1 - m0) / pooled


def differential_analysis(X, y, split_name="train"):
    numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    feature_cols = [c for c in numeric_cols if c != "RecordID"]

    records = []
    y = pd.Series(y).reset_index(drop=True)
    X_num = X[feature_cols].reset_index(drop=True)

    for feature in feature_cols:
        values = pd.to_numeric(X_num[feature], errors="coerce")
        x_surv = values[y == 0]
        x_death = values[y == 1]

        n_surv = int(x_surv.notna().sum())
        n_death = int(x_death.notna().sum())

        if n_surv < 2 or n_death < 2:
            t_stat, p_value, smd = np.nan, np.nan, np.nan
        else:
            t_stat, p_value = stats.ttest_ind(
                x_death.dropna(),
                x_surv.dropna(),
                equal_var=False,
                nan_policy="omit"
            )
            smd = compute_smd(x_surv, x_death)

        records.append({
            "feature": feature,
            "clinical_domain": clinical_domain(feature),
            "split": split_name,
            "n_survivors": n_surv,
            "n_deaths": n_death,
            "survivor_mean": float(x_surv.mean()) if n_surv else np.nan,
            "death_mean": float(x_death.mean()) if n_death else np.nan,
            "survivor_median": float(x_surv.median()) if n_surv else np.nan,
            "death_median": float(x_death.median()) if n_death else np.nan,
            "smd_death_minus_survivor": smd,
            "t_stat": t_stat,
            "p_value": p_value,
        })

    out = pd.DataFrame(records)
    out["fdr_bh"] = benjamini_hochberg(out["p_value"].values)
    out["abs_smd"] = out["smd_death_minus_survivor"].abs()
    out["direction"] = np.where(out["smd_death_minus_survivor"] > 0, "higher_in_deaths",
                         np.where(out["smd_death_minus_survivor"] < 0, "higher_in_survivors", "no_direction"))
    out["differential_supported"] = (
        (out["fdr_bh"] < DIFF_FDR_THRESHOLD) &
        (out["abs_smd"] >= DIFF_ABS_SMD_THRESHOLD)
    )
    out = out.sort_values(["differential_supported", "fdr_bh", "abs_smd"], ascending=[False, True, False])
    return out


train_diff = differential_analysis(X_train, y_train, split_name="train")
train_diff.to_csv(FIG3_RESULT_DIR / "differential_analysis_results_training_only.csv", index=False)

selected_diff = train_diff[train_diff["differential_supported"]].copy()
selected_diff.to_csv(FIG3_RESULT_DIR / "differential_selected_features_training_only.csv", index=False)

print(f"Tested features: {len(train_diff)}")
print(f"Selected differential features: {len(selected_diff)}")
selected_diff.head(20)


Tested features: 243
Selected differential features: 91


,feature,clinical_domain,split,n_survivors,n_deaths,survivor_mean,death_mean,survivor_median,death_median,smd_death_minus_survivor,t_stat,p_value,fdr_bh,abs_smd,direction,differential_supported
38,GCS_last,Neurological,train,6878,1122,12.875793,9.262839,15.000000,9.000000,-0.958434,-27.232793,2.598568e-130,6.132619e-128,0.958434,higher_in_survivors,True
41,GCS_median,Neurological,train,6878,1122,12.431537,9.519492,14.500000,9.000000,-0.780115,-22.504399,1.242324e-95,1.465943e-93,0.780115,higher_in_survivors,True
40,GCS_mean,Neurological,train,6878,1122,11.994649,9.464688,12.755952,9.267857,-0.737663,-21.161019,4.243690e-86,3.338369e-84,0.737663,higher_in_survivors,True
39,GCS_max,Neurological,train,6878,1122,13.829748,11.700086,15.000000,13.000000,-0.707754,-19.167047,4.170662e-72,2.460690e-70,0.707754,higher_in_survivors,True
2,BUN_last,Renal,train,6878,1122,23.348758,37.258644,18.000000,30.000000,0.618343,17.306077,1.303812e-60,6.153992e-59,0.618343,higher_in_deaths,True
4,BUN_mean,Renal,train,6878,1122,23.706060,36.477829,18.000000,29.000000,0.570449,16.076503,2.881691e-53,1.133465e-51,0.570449,higher_in_deaths,True
6,BUN_min,Renal,train,6878,1122,20.236735,31.964819,15.000000,25.000000,0.572716,15.994310,9.692148e-53,3.267639e-51,0.572716,higher_in_deaths,True
5,BUN_median,Renal,train,6878,1122,23.637322,36.334378,18.000000,28.750000,0.563152,15.894861,3.262422e-52,9.624145e-51,0.563152,higher_in_deaths,True
3,BUN_max,Renal,train,6878,1122,27.287154,41.223678,21.000000,34.000000,0.559806,15.866333,4.560714e-52,1.195921e-50,0.559806,higher_in_deaths,True
227,n_unique_parameters,Measurement / missingness,train,6878,1122,31.488659,33.688057,32.000000,34.000000,0.489140,15.735865,5.604564e-52,1.322677e-50,0.489140,higher_in_deaths,True


In [8]:
# ============================================================
# 8. Held-out train/test effect-size validation
# ============================================================

if y_test is not None:
    test_diff = differential_analysis(X_test, y_test, split_name="test")
    test_diff.to_csv(FIG3_RESULT_DIR / "differential_analysis_results_test_only_for_validation.csv", index=False)

    validation = (
        train_diff[["feature", "clinical_domain", "smd_death_minus_survivor", "fdr_bh", "abs_smd", "differential_supported"]]
        .rename(columns={
            "smd_death_minus_survivor": "train_smd",
            "fdr_bh": "train_fdr",
            "abs_smd": "train_abs_smd",
            "differential_supported": "train_differential_supported"
        })
        .merge(
            test_diff[["feature", "smd_death_minus_survivor", "fdr_bh", "abs_smd"]]
            .rename(columns={
                "smd_death_minus_survivor": "test_smd",
                "fdr_bh": "test_fdr",
                "abs_smd": "test_abs_smd"
            }),
            on="feature",
            how="left"
        )
    )

    validation["same_effect_direction"] = np.sign(validation["train_smd"]) == np.sign(validation["test_smd"])
    validation.loc[
        validation["train_smd"].isna() | validation["test_smd"].isna() |
        (validation["train_smd"] == 0) | (validation["test_smd"] == 0),
        "same_effect_direction"
    ] = np.nan

    selected_validation = validation[validation["train_differential_supported"]].copy()
    direction_consistency = selected_validation["same_effect_direction"].mean()

    validation.to_csv(FIG3_RESULT_DIR / "differential_train_test_validation.csv", index=False)
    selected_validation.to_csv(FIG3_RESULT_DIR / "differential_selected_train_test_validation.csv", index=False)

    top_replicated = selected_validation.sort_values(["train_abs_smd", "test_abs_smd"], ascending=False).head(30)
    top_replicated.to_csv(FIG3_RESULT_DIR / "figure3_top_replicated_differential_features.csv", index=False)

    print(f"Selected features with held-out validation: {len(selected_validation)}")
    print(f"Direction consistency among selected features: {direction_consistency:.3f}")
else:
    validation = train_diff.rename(columns={
        "smd_death_minus_survivor": "train_smd",
        "fdr_bh": "train_fdr",
        "abs_smd": "train_abs_smd",
        "differential_supported": "train_differential_supported"
    })
    validation["test_smd"] = np.nan
    validation["test_fdr"] = np.nan
    validation["test_abs_smd"] = np.nan
    validation["same_effect_direction"] = np.nan
    validation.to_csv(FIG3_RESULT_DIR / "differential_train_test_validation.csv", index=False)
    top_replicated = validation[validation["train_differential_supported"]].sort_values("train_abs_smd", ascending=False).head(30)
    top_replicated.to_csv(FIG3_RESULT_DIR / "figure3_top_replicated_differential_features.csv", index=False)
    direction_consistency = np.nan
    print("No y_test available; held-out validation skipped.")

top_replicated.head(20)


Selected features with held-out validation: 91
Direction consistency among selected features: 1.000


,feature,clinical_domain,train_smd,train_fdr,train_abs_smd,train_differential_supported,test_smd,test_fdr,test_abs_smd,same_effect_direction
0,GCS_last,Neurological,-0.958434,6.132619e-128,0.958434,True,-1.071110,1.101908e-79,1.071110,True
1,GCS_median,Neurological,-0.780115,1.465943e-93,0.780115,True,-0.888613,6.180880e-61,0.888613,True
2,GCS_mean,Neurological,-0.737663,3.338369e-84,0.737663,True,-0.839253,2.880255e-55,0.839253,True
3,GCS_max,Neurological,-0.707754,2.460690e-70,0.707754,True,-0.792146,1.797323e-44,0.792146,True
4,BUN_last,Renal,0.618343,6.153992e-59,0.618343,True,0.527011,6.109967e-23,0.527011,True
6,BUN_min,Renal,0.572716,3.267639e-51,0.572716,True,0.517912,3.239211e-22,0.517912,True
5,BUN_mean,Renal,0.570449,1.133465e-51,0.570449,True,0.502853,2.546971e-21,0.502853,True
7,BUN_median,Renal,0.563152,9.624145e-51,0.563152,True,0.505519,1.595532e-21,0.505519,True
8,BUN_max,Renal,0.559806,1.195921e-50,0.559806,True,0.478160,9.612149e-20,0.478160,True
12,BUN_first,Renal,0.500042,1.371677e-41,0.500042,True,0.445928,1.234033e-17,0.445928,True


In [9]:
# ============================================================
# 9. Prepare CoxPH model input
# ============================================================

def prepare_cox_training_input(X_train, cox_train):
    """
    Merge X_train features with Cox time/event info using RecordID.
    """
    if cox_train.empty:
        return pd.DataFrame(), "cox_train file not found or empty"

    needed = {"RecordID", "cox_time_days", "cox_event"}
    missing = needed - set(cox_train.columns)
    if missing:
        return pd.DataFrame(), f"cox_train missing required columns: {sorted(missing)}"

    if "RecordID" not in X_train.columns:
        return pd.DataFrame(), "X_train has no RecordID, and row-based Cox merge is not safe"

    cox_keep_cols = [
        c for c in [
            "RecordID", "set", "split", "SAPS_I", "SOFA",
            "Length_of_stay", "Survival",
            "in_hospital_death", "cox_event", "cox_time_days",
            "cox_time_source", "valid_for_cox", "cox_assumption"
        ]
        if c in cox_train.columns
    ]

    cx = cox_train[cox_keep_cols].copy()
    if "valid_for_cox" in cx.columns:
        cx = cx[cx["valid_for_cox"].astype(bool)].copy()

    merged = cx.merge(X_train, on="RecordID", how="inner")
    merged = merged[pd.to_numeric(merged["cox_time_days"], errors="coerce") > 0].copy()
    merged["cox_time_days"] = pd.to_numeric(merged["cox_time_days"], errors="coerce")
    merged["cox_event"] = pd.to_numeric(merged["cox_event"], errors="coerce").astype(int)

    return merged, None

cox_model_input, cox_input_error = prepare_cox_training_input(X_train, cox_train)

if cox_input_error:
    print("CoxPH input issue:", cox_input_error)
    with open(FIG3_RESULT_DIR / "coxph_skipped_reason.txt", "w", encoding="utf-8") as f:
        f.write(cox_input_error)
else:
    print(f"CoxPH model input shape: {cox_model_input.shape}")
    print(f"CoxPH events: {int(cox_model_input['cox_event'].sum())} / {len(cox_model_input)}")
    cox_model_input.to_csv(FIG3_RESULT_DIR / "coxph_model_input_train_ab.csv", index=False)

cox_model_input.head()


CoxPH input issue: X_train has no RecordID, and row-based Cox merge is not safe


""


In [10]:
# ============================================================
# 10. Select CoxPH candidate features
# ============================================================

def get_shared_top30_features(top30_shared, top30_rankings):
    """
    Prefer explicit shared-feature file; fallback to features that appear in >=2 model top-30 lists.
    """
    if not top30_shared.empty and "feature" in top30_shared.columns:
        out = top30_shared.copy()
        if "mean_rank" not in out.columns:
            rank_cols = [c for c in out.columns if "rank" in c.lower()]
            if rank_cols:
                out["mean_rank"] = out[rank_cols].mean(axis=1)
            else:
                out["mean_rank"] = np.arange(1, len(out) + 1)
        return out.sort_values("mean_rank")

    if not top30_rankings.empty and {"model_name", "feature", "rank"}.issubset(top30_rankings.columns):
        tmp = top30_rankings.copy()
        tmp = tmp[tmp["rank"] <= 30]
        grouped = tmp.groupby("feature").agg(
            n_models=("model_name", "nunique"),
            mean_rank=("rank", "mean")
        ).reset_index()
        grouped = grouped[grouped["n_models"] >= 2].sort_values("mean_rank")
        return grouped

    return pd.DataFrame(columns=["feature", "mean_rank"])

shared_features_df = get_shared_top30_features(top30_shared, top30_rankings)

shared_features = shared_features_df["feature"].dropna().astype(str).tolist()
top_diff_features = top_replicated["feature"].dropna().astype(str).tolist() if "top_replicated" in globals() else []

candidate_features = []
for f in shared_features + top_diff_features:
    if f not in candidate_features and f in X_train.columns:
        candidate_features.append(f)

candidate_features = candidate_features[:40]

cox_candidates = pd.DataFrame({
    "feature": candidate_features,
    "clinical_domain": [clinical_domain(f) for f in candidate_features],
    "candidate_source": [
        ("shared_top30_ml" if f in shared_features else "top_differential")
        for f in candidate_features
    ]
})
cox_candidates.to_csv(FIG3_RESULT_DIR / "coxph_candidate_features.csv", index=False)

print(f"CoxPH candidate features: {len(cox_candidates)}")
cox_candidates.head(40)


CoxPH candidate features: 36


,feature,clinical_domain,candidate_source
0,GCS_last,Neurological,shared_top30_ml
1,BUN_last,Renal,shared_top30_ml
2,static_Age,Static / demographic,shared_top30_ml
3,Lactate_last,Metabolic,shared_top30_ml
4,Urine_mean,Renal,shared_top30_ml
5,GCS_mean,Neurological,shared_top30_ml
6,Urine_total,Renal,shared_top30_ml
7,Creatinine_median,Renal,shared_top30_ml
8,GCS_std,Neurological,shared_top30_ml
9,PaO2_median,Respiratory,shared_top30_ml


In [11]:
# ============================================================
# 11. Run univariate CoxPH for candidate features
# ============================================================

def fit_univariate_cox_lifelines(df, feature):
    work = df[["cox_time_days", "cox_event", feature]].copy()
    work[feature] = pd.to_numeric(work[feature], errors="coerce")
    work = work.replace([np.inf, -np.inf], np.nan).dropna()

    if work.shape[0] < 50 or work["cox_event"].sum() < 10:
        return None, "too few valid rows/events"

    if work[feature].nunique(dropna=True) < 2:
        return None, "constant feature"

    sd = work[feature].std(ddof=0)
    if not np.isfinite(sd) or sd == 0:
        return None, "zero sd"
    work[feature] = (work[feature] - work[feature].mean()) / sd

    cph = CoxPHFitter()
    cph.fit(work, duration_col="cox_time_days", event_col="cox_event")
    s = cph.summary.loc[feature]
    return {
        "feature": feature,
        "n": int(work.shape[0]),
        "n_events": int(work["cox_event"].sum()),
        "coef": float(s["coef"]),
        "se_coef": float(s["se(coef)"]),
        "hazard_ratio": float(s["exp(coef)"]),
        "ci_lower": float(s["exp(coef) lower 95%"]),
        "ci_upper": float(s["exp(coef) upper 95%"]),
        "z": float(s["z"]),
        "p_value": float(s["p"]),
        "cox_backend": "lifelines",
        "scale": "per_1_sd"
    }, None


def fit_univariate_cox_phreg(df, feature):
    work = df[["cox_time_days", "cox_event", feature]].copy()
    work[feature] = pd.to_numeric(work[feature], errors="coerce")
    work = work.replace([np.inf, -np.inf], np.nan).dropna()

    if work.shape[0] < 50 or work["cox_event"].sum() < 10:
        return None, "too few valid rows/events"

    if work[feature].nunique(dropna=True) < 2:
        return None, "constant feature"

    sd = work[feature].std(ddof=0)
    if not np.isfinite(sd) or sd == 0:
        return None, "zero sd"
    x = ((work[feature] - work[feature].mean()) / sd).astype(float)

    try:
        model = PHReg(
            endog=work["cox_time_days"].astype(float),
            exog=x.to_frame(feature),
            status=work["cox_event"].astype(int)
        )
        result = model.fit(disp=False)
        coef = float(result.params[0])
        se = float(result.bse[0])
        pval = float(result.pvalues[0])
        ci = result.conf_int()
        lo, hi = float(ci[0, 0]), float(ci[0, 1])
        return {
            "feature": feature,
            "n": int(work.shape[0]),
            "n_events": int(work["cox_event"].sum()),
            "coef": coef,
            "se_coef": se,
            "hazard_ratio": float(np.exp(coef)),
            "ci_lower": float(np.exp(lo)),
            "ci_upper": float(np.exp(hi)),
            "z": float(coef / se) if se else np.nan,
            "p_value": pval,
            "cox_backend": "statsmodels_PHReg",
            "scale": "per_1_sd"
        }, None
    except Exception as e:
        return None, str(e)


cox_records = []
cox_errors = []

if cox_model_input.empty:
    reason = cox_input_error or "Cox model input is empty"
    print("CoxPH skipped:", reason)
    with open(FIG3_RESULT_DIR / "coxph_skipped_reason.txt", "w", encoding="utf-8") as f:
        f.write(reason)
elif len(candidate_features) == 0:
    reason = "No CoxPH candidate features available."
    print("CoxPH skipped:", reason)
    with open(FIG3_RESULT_DIR / "coxph_skipped_reason.txt", "w", encoding="utf-8") as f:
        f.write(reason)
elif not HAS_LIFELINES and not HAS_PHREG:
    reason = "No CoxPH backend available. Install lifelines or statsmodels."
    print("CoxPH skipped:", reason)
    with open(FIG3_RESULT_DIR / "coxph_skipped_reason.txt", "w", encoding="utf-8") as f:
        f.write(reason)
else:
    for feature in candidate_features:
        if HAS_LIFELINES:
            rec, err = fit_univariate_cox_lifelines(cox_model_input, feature)
        else:
            rec, err = fit_univariate_cox_phreg(cox_model_input, feature)

        if rec is not None:
            rec["clinical_domain"] = clinical_domain(feature)
            cox_records.append(rec)
        else:
            cox_errors.append({"feature": feature, "error": err})

    cox_results = pd.DataFrame(cox_records)
    cox_errors_df = pd.DataFrame(cox_errors)

    if not cox_results.empty:
        cox_results["fdr_bh"] = benjamini_hochberg(cox_results["p_value"].values)
        cox_results["coxph_supported"] = cox_results["fdr_bh"] < COX_FDR_THRESHOLD
        cox_results = cox_results.sort_values(["coxph_supported", "fdr_bh", "hazard_ratio"], ascending=[False, True, False])
    else:
        cox_results = pd.DataFrame(columns=[
            "feature", "clinical_domain", "n", "n_events", "coef", "se_coef", 
            "hazard_ratio", "ci_lower", "ci_upper", "z", "p_value", "fdr_bh",
            "coxph_supported", "cox_backend", "scale"
        ])

    cox_results.to_csv(FIG3_RESULT_DIR / "coxph_univariate_results.csv", index=False)
    cox_errors_df.to_csv(FIG3_RESULT_DIR / "coxph_univariate_errors.csv", index=False)

    print(f"CoxPH successful features: {len(cox_results)}")
    print(f"CoxPH failed features: {len(cox_errors_df)}")

if "cox_results" not in globals():
    cox_results = pd.DataFrame(columns=[
        "feature", "clinical_domain", "n", "n_events", "coef", "se_coef",
        "hazard_ratio", "ci_lower", "ci_upper", "z", "p_value", "fdr_bh",
        "coxph_supported", "cox_backend", "scale"
    ])
    cox_results.to_csv(FIG3_RESULT_DIR / "coxph_univariate_results.csv", index=False)

cox_results.head(20)


CoxPH skipped: X_train has no RecordID, and row-based Cox merge is not safe


,feature,clinical_domain,n,n_events,coef,se_coef,hazard_ratio,ci_lower,ci_upper,z,p_value,fdr_bh,coxph_supported,cox_backend,scale


In [12]:
# ============================================================
# 12. Build Figure 3 evidence support matrix
# ============================================================

if not shared_features_df.empty:
    support = shared_features_df.copy()
    support["ml_shared_top30"] = True
else:
    support = pd.DataFrame({"feature": candidate_features})
    support["ml_shared_top30"] = False
    support["mean_rank"] = np.arange(1, len(support) + 1)

if not top30_rankings.empty and {"model_name", "feature", "rank"}.issubset(top30_rankings.columns):
    rank_pivot = (
        top30_rankings
        .pivot_table(index="feature", columns="model_name", values="rank", aggfunc="min")
        .reset_index()
    )
    rank_pivot.columns = [
        "feature" if c == "feature" else f"rank_{str(c).replace(' ', '_').replace('-', '_')}"
        for c in rank_pivot.columns
    ]
    support = support.merge(rank_pivot, on="feature", how="left")

diff_support = validation[[
    "feature", "clinical_domain", "train_smd", "train_fdr", "train_abs_smd",
    "train_differential_supported", "test_smd", "test_fdr", "test_abs_smd",
    "same_effect_direction"
]].copy()

support = support.merge(diff_support, on="feature", how="left")

if not cox_results.empty:
    cox_support = cox_results[[
        "feature", "hazard_ratio", "ci_lower", "ci_upper", "p_value", "fdr_bh", "coxph_supported"
    ]].rename(columns={
        "hazard_ratio": "cox_hr",
        "ci_lower": "cox_ci_lower",
        "ci_upper": "cox_ci_upper",
        "p_value": "cox_p_value",
        "fdr_bh": "cox_fdr"
    })
else:
    cox_support = pd.DataFrame(columns=[
        "feature", "cox_hr", "cox_ci_lower", "cox_ci_upper", "cox_p_value", "cox_fdr", "coxph_supported"
    ])

support = support.merge(cox_support, on="feature", how="left")

if "clinical_domain" not in support.columns:
    support["clinical_domain"] = support["feature"].map(clinical_domain)
else:
    support["clinical_domain"] = support["clinical_domain"].fillna(support["feature"].map(clinical_domain))

support["train_differential_supported"] = support["train_differential_supported"].fillna(False).astype(bool)
support["heldout_same_direction"] = support["same_effect_direction"]
support["coxph_supported"] = support["coxph_supported"].fillna(False).astype(bool)

support["n_support_sources"] = (
    support["ml_shared_top30"].fillna(False).astype(int)
    + support["train_differential_supported"].fillna(False).astype(int)
    + support["heldout_same_direction"].fillna(False).astype(int)
    + support["coxph_supported"].fillna(False).astype(int)
)

sort_cols = ["n_support_sources"]
ascending = [False]
if "mean_rank" in support.columns:
    sort_cols.append("mean_rank")
    ascending.append(True)
if "train_abs_smd" in support.columns:
    sort_cols.append("train_abs_smd")
    ascending.append(False)

support = support.sort_values(sort_cols, ascending=ascending)

support.to_csv(FIG3_RESULT_DIR / "figure3_evidence_support_matrix.csv", index=False)
support.head(20).to_csv(FIG3_RESULT_DIR / "figure3_evidence_support_matrix_top20.csv", index=False)

print(f"Evidence support matrix rows: {len(support)}")
support.head(20)


Evidence support matrix rows: 12


,model_a,model_b,feature,rank_a,rank_b,mean_rank,ml_shared_top30,rank_logistic_regression,rank_xgboost_auc_optimized,clinical_domain,...,test_abs_smd,same_effect_direction,cox_hr,cox_ci_lower,cox_ci_upper,cox_p_value,cox_fdr,coxph_supported,heldout_same_direction,n_support_sources
0,logistic_regression,xgboost_auc_optimized,GCS_last,1,1,1.0,True,1.0,1.0,Neurological,...,1.071110,True,NaN,NaN,NaN,NaN,NaN,False,True,3
1,logistic_regression,xgboost_auc_optimized,BUN_last,2,2,2.0,True,2.0,2.0,Renal,...,0.527011,True,NaN,NaN,NaN,NaN,NaN,False,True,3
2,logistic_regression,xgboost_auc_optimized,static_Age,4,9,6.5,True,4.0,9.0,Static / demographic,...,0.357503,True,NaN,NaN,NaN,NaN,NaN,False,True,3
3,logistic_regression,xgboost_auc_optimized,Lactate_last,8,11,9.5,True,8.0,11.0,Metabolic,...,0.359000,True,NaN,NaN,NaN,NaN,NaN,False,True,3
4,logistic_regression,xgboost_auc_optimized,Urine_mean,6,14,10.0,True,6.0,14.0,Renal,...,0.526179,True,NaN,NaN,NaN,NaN,NaN,False,True,3
5,logistic_regression,xgboost_auc_optimized,GCS_mean,3,19,11.0,True,3.0,19.0,Neurological,...,0.839253,True,NaN,NaN,NaN,NaN,NaN,False,True,3
6,logistic_regression,xgboost_auc_optimized,Urine_total,22,4,13.0,True,22.0,4.0,Renal,...,0.447614,True,NaN,NaN,NaN,NaN,NaN,False,True,3
7,logistic_regression,xgboost_auc_optimized,Creatinine_median,21,8,14.5,True,21.0,8.0,Renal,...,0.226004,True,NaN,NaN,NaN,NaN,NaN,False,True,3
9,logistic_regression,xgboost_auc_optimized,PaO2_median,17,30,23.5,True,17.0,30.0,Respiratory,...,0.248787,True,NaN,NaN,NaN,NaN,NaN,False,True,3
11,logistic_regression,xgboost_auc_optimized,HR_mean,29,22,25.5,True,29.0,22.0,Cardiovascular,...,0.208298,True,NaN,NaN,NaN,NaN,NaN,False,True,3


In [13]:
# ============================================================
# 13. Save compact calculation summary
# ============================================================

summary_records = []

summary_records.append({"metric": "n_train_samples", "value": int(len(X_train))})
summary_records.append({"metric": "n_test_samples", "value": int(len(X_test))})
summary_records.append({"metric": "n_original_train_columns", "value": int(X_train.shape[1])})
summary_records.append({"metric": "train_mortality_rate", "value": float(y_train.mean())})
summary_records.append({"metric": "test_mortality_rate", "value": float(y_test.mean()) if y_test is not None else np.nan})
summary_records.append({"metric": "n_tested_differential_features", "value": int(len(train_diff))})
summary_records.append({"metric": "n_selected_differential_features", "value": int(len(selected_diff))})
summary_records.append({"metric": "differential_fdr_threshold", "value": DIFF_FDR_THRESHOLD})
summary_records.append({"metric": "differential_abs_smd_threshold", "value": DIFF_ABS_SMD_THRESHOLD})
summary_records.append({"metric": "heldout_direction_consistency_selected_features", "value": float(direction_consistency) if pd.notna(direction_consistency) else np.nan})
summary_records.append({"metric": "n_top30_shared_features", "value": int(len(shared_features_df))})
summary_records.append({"metric": "n_cox_valid_train_rows", "value": int(len(cox_model_input)) if not cox_model_input.empty else 0})
summary_records.append({"metric": "n_cox_events_train", "value": int(cox_model_input["cox_event"].sum()) if not cox_model_input.empty else 0})
summary_records.append({"metric": "n_cox_candidate_features", "value": int(len(candidate_features))})
summary_records.append({"metric": "n_cox_successful_features", "value": int(len(cox_results))})
summary_records.append({"metric": "n_cox_supported_features_fdr_lt_0_05", "value": int(cox_results["coxph_supported"].sum()) if not cox_results.empty and "coxph_supported" in cox_results.columns else 0})
summary_records.append({"metric": "cox_backend_used", "value": str(cox_results["cox_backend"].dropna().iloc[0]) if not cox_results.empty and "cox_backend" in cox_results.columns and len(cox_results["cox_backend"].dropna()) > 0 else "none"})

summary = pd.DataFrame(summary_records)
summary.to_csv(FIG3_RESULT_DIR / "figure3_calculation_summary.csv", index=False)

readme = f"""
Figure 3 calculation outputs

Output folder:
{FIG3_RESULT_DIR}

Main files for the next figure-making notebook:
- differential_analysis_results_training_only.csv
- differential_selected_features_training_only.csv
- differential_train_test_validation.csv
- figure3_top_replicated_differential_features.csv
- coxph_candidate_features.csv
- coxph_univariate_results.csv
- figure3_evidence_support_matrix.csv
- figure3_evidence_support_matrix_top20.csv
- figure3_calculation_summary.csv

No figure files were created by this notebook.
"""
with open(FIG3_RESULT_DIR / "README_figure3_calculation_outputs.txt", "w", encoding="utf-8") as f:
    f.write(readme.strip())

print("Saved all calculation outputs to:")
print(FIG3_RESULT_DIR)
summary


Saved all calculation outputs to:
C:\Users\junse\Documents\research\IUBDC 2026\differential_analysis\result\figure3_calculation


,metric,value
0,n_train_samples,8000
1,n_test_samples,4000
2,n_original_train_columns,243
3,train_mortality_rate,0.14025
4,test_mortality_rate,0.14625
5,n_tested_differential_features,243
6,n_selected_differential_features,91
7,differential_fdr_threshold,0.05
8,differential_abs_smd_threshold,0.2
9,heldout_direction_consistency_selected_features,1.0


## Done

The notebook is complete once all cells above run successfully.

The next notebook can focus only on figure generation using the CSV files saved in `result/figure3_calculation/`.
